### Smart-Stock TrOCR — Kaggle

### Setup & Paths (pip installs + env)

In [1]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q transformers==5.0.0 datasets evaluate jiwer albumentations
!pip install -q "peft==0.13.2"
!pip install -q "torchao>=0.16.0"
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.0 MB/s eta 0:00:00:00:01


In [10]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # MUST be before ALL imports — especially before torch

from pathlib import Path

INPUT_DIR   = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

# v3 dataset (CORD + SROIE + WildReceipt) — loads from disk if exists, builds if not
DATASET_DIR = Path("/kaggle/input/datasets/maazahmad69/smart-stock-dataset-v3/smart_stock_dataset_v3")

# WildReceipt raw input — only needed during dataset build cell
WILDRECEIPT_DIR = Path("/kaggle/input/datasets/maazahmad69/wild-receipt/wildreceipt")

# Best model weights — loaded as base for fine-tuning
MODEL_INPUT = Path("/kaggle/input/datasets/maazahmad69/trocr-smart-stock-best/content/drive/MyDrive/SmartStock/trocr-smart-stock-best")

# Checkpoint dir from previously uploaded dataset (fallback resume source)
INPUT_CHECKPOINT_DIR = Path("/kaggle/input/smart-stock-dataset/trocr-smart-stock")

# Output dirs — writable
CHECKPOINT_DIR = WORKING_DIR / "trocr-smart-stock"
BEST_MODEL_DIR = WORKING_DIR / "trocr-smart-stock-best"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset dir     : {DATASET_DIR}")
print(f"WildReceipt dir : {WILDRECEIPT_DIR}")
print(f"Model input     : {MODEL_INPUT}")
print(f"Checkpoints     : {CHECKPOINT_DIR}")
print(f"Best model      : {BEST_MODEL_DIR}")

# Verify model files present
for f in ["config.json", "model.safetensors", "generation_config.json"]:
    exists = (MODEL_INPUT / f).exists()
    print(f"  {'✅' if exists else '❌'} {f}")

Dataset dir     : /kaggle/input/datasets/maazahmad69/smart-stock-dataset-v3/smart_stock_dataset_v3
WildReceipt dir : /kaggle/input/datasets/maazahmad69/wild-receipt/wildreceipt
Model input     : /kaggle/input/datasets/maazahmad69/trocr-smart-stock-best/content/drive/MyDrive/SmartStock/trocr-smart-stock-best
Checkpoints     : /kaggle/working/trocr-smart-stock
Best model      : /kaggle/working/trocr-smart-stock-best
  ✅ config.json
  ✅ model.safetensors
  ✅ generation_config.json


#### **CORD's ground_truth** field is a raw JSON string. Parse it and construct a flat text target from the menu array.

In [11]:
import json
from collections import defaultdict
from PIL import Image

def extract_cord_crops(image: Image.Image, ground_truth_str: str) -> list:
    """
    Extract line-level crops from a CORD receipt image.

    CORD's ground_truth is a raw JSON string. Bounding boxes are in
    valid_line[].words[].quad — NOT in gt_parse.menu.
    Each group_id in valid_line = one logical receipt line.
    We group words by group_id, merge their quads into one bbox, crop it,
    and use the joined word texts as the OCR target.
    Only menu.* categories are kept (skip total, tax, header lines).

    Returns list of (cropped_PIL_image, text) pairs.
    One CORD receipt → ~8–12 crops.
    """
    try:
        data = json.loads(ground_truth_str)
    except (json.JSONDecodeError, AttributeError):
        return []

    valid_lines = data.get("valid_line", [])
    if not valid_lines:
        return []

    groups = defaultdict(list)
    for line in valid_lines:
        groups[line["group_id"]].append(line)

    w, h = image.size
    crops = []

    for gid, lines in groups.items():
        if not any(l.get("category", "").startswith("menu.") for l in lines):
            continue

        all_words, all_xs, all_ys = [], [], []
        for line in lines:
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                if text:
                    all_words.append(text)
                q = word.get("quad", {})
                if q:
                    all_xs += [q.get("x1",0), q.get("x2",0), q.get("x3",0), q.get("x4",0)]
                    all_ys += [q.get("y1",0), q.get("y2",0), q.get("y3",0), q.get("y4",0)]

        if not all_words or not all_xs:
            continue

        text = " ".join(all_words)
        x1, y1 = max(0, min(all_xs)), max(0, min(all_ys))
        x2, y2 = min(w, max(all_xs)), min(h, max(all_ys))
        if x2 <= x1 or y2 <= y1:
            continue

        crops.append((image.crop((x1, y1, x2, y2)), text))

    return crops

### **SROIE** Text Reconstruction
#### SROIE has bboxes per word. Group words by Y-coordinate (within 15px = same line), merge bounding boxes per line, crop each line region.

In [12]:
def extract_sroie_crops(image: Image.Image, words: list, bboxes: list) -> list:
    """
    Group SROIE words into lines by Y-coordinate proximity (15px tolerance).
    Each line bbox is cropped and paired with its joined text.
    SROIE has bboxes per word — no pre-grouped lines.

    Returns list of (cropped_PIL_image, text) pairs.
    """
    if not words or not bboxes:
        return []

    items = sorted(zip(words, bboxes), key=lambda x: x[1][1])

    lines = []
    current_words, current_boxes = [items[0][0]], [items[0][1]]
    for word, box in items[1:]:
        if abs(box[1] - current_boxes[-1][1]) <= 15:
            current_words.append(word)
            current_boxes.append(box)
        else:
            lines.append((current_words, current_boxes))
            current_words, current_boxes = [word], [box]
    lines.append((current_words, current_boxes))

    w, h = image.size
    crops = []
    for line_words, line_boxes in lines:
        text = " ".join(line_words).strip()
        if not text:
            continue
        x1 = max(0, min(b[0] for b in line_boxes))
        y1 = max(0, min(b[1] for b in line_boxes))
        x2 = min(w, max(b[2] for b in line_boxes))
        y2 = min(h, max(b[3] for b in line_boxes))
        if x2 <= x1 or y2 <= y1:
            continue
        crops.append((image.crop((x1, y1, x2, y2)), text))
    return crops

### WildReceipt Extractor 

In [13]:
# Labels to EXCLUDE from WildReceipt:
# 0  = empty string / illegible text
# 25 = catch-all "other" (terminal IDs, legal footnotes, thank-you messages, promo text)
# All other labels (1=store name, 3=address, 5=phone, 7=date, 9=time,
# 11=item name, 13=quantity, 14=count, 15=item price, 17=subtotal,
# 18=subtotal label, 19=tax amount, 20=tax label, 22=tip, 23=total,
# 24=total label) are included.
import json

WILDRECEIPT_EXCLUDE = {0, 25}

def extract_wildreceipt_crops(annotation_file: Path) -> list:
    """
    Parse a WildReceipt annotation file (one JSON object per line).
    Each annotation becomes its own crop — no line grouping.

    This avoids the two-column merging problem where item names (left column)
    and prices (right column) share nearly identical Y coordinates and would
    incorrectly merge under Y-proximity grouping.

    Each annotation box: [x1,y1, x2,y1, x2,y2, x1,y2] clockwise from top-left.
    Returns list of (PIL.Image, text) pairs.
    """
    crops = []

    with open(annotation_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            img_path = WILDRECEIPT_DIR / record["file_name"]
            if not img_path.exists():
                continue

            try:
                image = Image.open(img_path).convert("RGB")
            except Exception:
                continue

            img_w, img_h = image.size

            for ann in record["annotations"]:
                # Skip excluded labels and empty text
                if ann["label"] in WILDRECEIPT_EXCLUDE:
                    continue
                text = ann.get("text", "").strip()
                if not text:
                    continue

                # Box: [x1,y1, x2,y1, x2,y2, x1,y2]
                box = ann["box"]
                xs = [box[0], box[2], box[4], box[6]]
                ys = [box[1], box[3], box[5], box[7]]

                x1 = max(0, int(min(xs)))
                y1 = max(0, int(min(ys)))
                x2 = min(img_w, int(max(xs)))
                y2 = min(img_h, int(max(ys)))

                # Skip degenerate crops — beam search hangs on these
                if x2 <= x1 or y2 <= y1:
                    continue
                if (x2 - x1) < 4 or (y2 - y1) < 4:
                    continue

                crops.append((image.crop((x1, y1, x2, y2)), text))

    return crops

### Combined Dataset Builder
OOM fix: Images encoded to PNG bytes immediately. Line crops: Each receipt yields multiple (crop, text) pairs. SROIE 2x weighting: SROIE train concatenated twice.

In [14]:
import io
import json
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from PIL import Image

# ── Helpers ───────────────────────────────────────────────────────────────────

def pil_to_bytes(img: Image.Image) -> bytes:
    """Encode PIL image to PNG bytes immediately — prevents RAM accumulation."""
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="PNG")
    return buf.getvalue()

def iter_to_dataset(iterator) -> Dataset:
    """
    Convert iterator of (PIL Image, text) tuples into a HuggingFace Dataset.
    Encodes each image to bytes immediately — never holds multiple PIL objects in RAM.
    """
    img_bytes, texts = [], []
    for img, text in iterator:
        img_bytes.append(pil_to_bytes(img))
        texts.append(text)
    return Dataset.from_dict({"image_bytes": img_bytes, "text": texts})

# ── Combined dataset builder ──────────────────────────────────────────────────

DATASET_SAVE = DATASET_DIR  # v3 path — loads if exists, builds if not

WR_TEST_KEEP = 1000  # cap WildReceipt test crops; remainder folds into train

def build_and_save_dataset():
    if DATASET_SAVE.exists():
        print(f"Dataset found at {DATASET_SAVE} — loading from disk...")
        return DatasetDict.load_from_disk(str(DATASET_SAVE))

    print("Building v3 line-crop dataset (CORD + SROIE + WildReceipt)...")

    # ── CORD ──────────────────────────────────────────────────────────────────
    print("Loading CORD...")
    cord = load_dataset("naver-clova-ix/cord-v2")

    def cord_iter(split):
        for ex in cord[split]:
            for crop, text in extract_cord_crops(ex["image"], ex["ground_truth"]):
                yield crop, text

    cord_train      = iter_to_dataset(cord_iter("train"))
    cord_validation = iter_to_dataset(cord_iter("validation"))
    cord_test       = iter_to_dataset(cord_iter("test"))
    print(f"  CORD  train: {len(cord_train)} | val: {len(cord_validation)} | test: {len(cord_test)}")
    del cord  # free RAM before loading next dataset

    # ── SROIE ─────────────────────────────────────────────────────────────────
    print("Loading SROIE...")
    sroie = load_dataset("sizhkhy/SROIE")

    def sroie_iter(split):
        for ex in sroie[split]:
            for crop, text in extract_sroie_crops(ex["images"], ex["words"], ex["bboxes"]):
                yield crop, text

    sroie_train = iter_to_dataset(sroie_iter("train"))
    sroie_test  = iter_to_dataset(sroie_iter("test"))
    print(f"  SROIE train: {len(sroie_train)} | test: {len(sroie_test)}")
    del sroie

    # ── WildReceipt ───────────────────────────────────────────────────────────
    print("Loading WildReceipt...")
    wr_train_crops = extract_wildreceipt_crops(WILDRECEIPT_DIR / "train.txt")
    wr_test_crops  = extract_wildreceipt_crops(WILDRECEIPT_DIR / "test.txt")

    wr_train_raw = iter_to_dataset(iter(wr_train_crops))
    wr_test_raw  = iter_to_dataset(iter(wr_test_crops))
    print(f"  WildReceipt raw — train: {len(wr_train_raw)} | test: {len(wr_test_raw)}")

    # 90% train / 10% val split from WildReceipt train.txt crops
    wr_train_split = wr_train_raw.train_test_split(test_size=0.1, seed=42)
    wr_train_final = wr_train_split["train"]
    wr_val         = wr_train_split["test"]
    print(f"  WildReceipt train (after val split): {len(wr_train_final)} | val: {len(wr_val)}")

    # Cap test.txt crops at WR_TEST_KEEP (1000), move remainder to train
    # Reason: 5,103 test crops * beam search = hours of eval time (killed prior session)
    if len(wr_test_raw) > WR_TEST_KEEP:
        wr_test_split    = wr_test_raw.train_test_split(test_size=WR_TEST_KEEP, seed=42)
        wr_test_final    = wr_test_split["test"]
        wr_test_to_train = wr_test_split["train"]
    else:
        wr_test_final    = wr_test_raw
        wr_test_to_train = None

    print(f"  WildReceipt test kept: {len(wr_test_final)} | moved to train: {len(wr_test_to_train) if wr_test_to_train else 0}")

    # ── Combine ───────────────────────────────────────────────────────────────
    # SROIE 2x weighted in train — more English receipt signal vs CORD's Indonesian
    # Excess WildReceipt test crops folded into train (free data)
    train_parts = [cord_train, sroie_train, sroie_train, wr_train_final]
    if wr_test_to_train:
        train_parts.append(wr_test_to_train)

    dataset_dict = DatasetDict({
        "train":      concatenate_datasets(train_parts),
        "validation": concatenate_datasets([cord_validation, wr_val]),
        "test":       concatenate_datasets([cord_test, sroie_test, wr_test_final]),
    })

    # Save to working dir — download and upload as Kaggle dataset smart-stock-dataset-v3
    save_path = WORKING_DIR / "smart_stock_dataset_v3"
    dataset_dict.save_to_disk(str(save_path))
    print(f"\n✅ Saved to {save_path}")
    print(f"   Train      : {len(dataset_dict['train'])}")
    print(f"   Validation : {len(dataset_dict['validation'])}")
    print(f"   Test       : {len(dataset_dict['test'])}")
    return dataset_dict

combined_dataset = build_and_save_dataset()

Building v3 line-crop dataset (CORD + SROIE + WildReceipt)...
Loading CORD...
  CORD  train: 2105 | val: 221 | test: 251
Loading SROIE...
  SROIE train: 14476 | test: 8050
Loading WildReceipt...
  WildReceipt raw — train: 29713 | test: 11665
  WildReceipt train (after val split): 26741 | val: 2972
  WildReceipt test kept: 1000 | moved to train: 10665


Saving the dataset (0/2 shards):   0%|          | 0/68463 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3193 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9301 [00:00<?, ? examples/s]


✅ Saved to /kaggle/working/smart_stock_dataset_v3
   Train      : 68463
   Validation : 3193
   Test       : 9301


In [15]:
# import json
# import io
# from PIL import Image

# # Pick 3 WildReceipt images and show crop count + text per crop
# with open(WILDRECEIPT_DIR / "train.txt") as f:
#     lines = [l.strip() for l in f if l.strip()][:3]

# for line in lines:
#     record = json.loads(line)
#     img = Image.open(WILDRECEIPT_DIR / record["file_name"]).convert("RGB")
#     img_w, img_h = img.size
#     y_tol = max(10, int(img_h * 0.02))
    
#     valid = [a for a in record["annotations"]
#              if a["label"] not in {0, 25} and a.get("text","").strip()]
    
#     print(f"\nImage: {record['file_name']} ({img_w}×{img_h}), y_tol={y_tol}")
#     print(f"Valid annotations: {len(valid)}")
#     for a in valid:
#         box = a["box"]
#         ys = [box[1], box[3], box[5], box[7]]
#         cy = (min(ys) + max(ys)) / 2
#         print(f"  cy={cy:.0f}  text='{a['text']}'  label={a['label']}")

#### **Augmentation**
Apply augmentation only to training images, inline during the preprocess_trocr step.

In [7]:
import albumentations as A
import cv2
import numpy as np
from PIL import Image, ImageOps

receipt_augmentation = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.4, 0.15), p=0.6),  # thermal fade simulation
    A.GaussNoise(p=0.5),                                                 # scanner noise
    A.Rotate(limit=8, border_mode=cv2.BORDER_REPLICATE, p=0.5),         # crumple/tilt
    A.Perspective(scale=(0.02, 0.08), p=0.4),                           # phone photo angle
    A.MotionBlur(blur_limit=5, p=0.3),                                  # shaky photo
    A.ImageCompression(p=0.5),                                           # JPEG artifact
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),                           # focus blur
    A.RandomShadow(p=0.2),                                               # shadow on receipt
])

def apply_augmentation(pil_image: Image.Image) -> Image.Image:
    """Pad tiny line crops to min 32px height, then augment."""
    pil_image = pil_image.convert("RGB")
    if pil_image.height < 32:
        pad = 32 - pil_image.height
        pil_image = ImageOps.expand(pil_image, border=(0, pad//2, 0, pad - pad//2), fill=255)
    img_np = np.array(pil_image)
    augmented = receipt_augmentation(image=img_np)["image"]
    return Image.fromarray(augmented)

### Preprocessing Function

In [8]:
import io
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

def preprocess_trocr(example, augment: bool = False):
    """
    Preprocess a single (image_bytes, text) example at access time.
    Returns dict with pixel_values (tensor) and labels (tensor).
    Padding tokens replaced with -100 so loss ignores them.
    """
    image = Image.open(io.BytesIO(example["image_bytes"])).convert("RGB")

    if augment:
        image = apply_augmentation(image)

    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    labels = processor.tokenizer(
        example["text"],
        padding="max_length",
        max_length=128,
        truncation=True,
    ).input_ids

    labels = [
        t if t != processor.tokenizer.pad_token_id else -100
        for t in labels
    ]

    return {
        "pixel_values": pixel_values.squeeze(),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


class TrOCRDataset(TorchDataset):
    """
    On-the-fly preprocessing — processes each example at access time.
    Replaces .map() which caused either:
      - OOM (keep_in_memory=True on 46k examples)
      - OSError Errno 28 (cache writing to read-only /kaggle/input/)
    Zero disk usage, zero RAM accumulation, full dataset used.
    Compatible with Seq2SeqTrainer.

    NOTE: This is a PyTorch Dataset — it has no .select() method.
    For Optuna subset selection, use combined_dataset["train"].select(...)
    (the HuggingFace dataset) and then wrap in TrOCRDataset().
    """
    def __init__(self, hf_dataset, augment: bool = False):
        self.data    = hf_dataset
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return preprocess_trocr(self.data[idx], augment=self.augment)


train_dataset = TrOCRDataset(combined_dataset["train"],      augment=True)
val_dataset   = TrOCRDataset(combined_dataset["validation"], augment=False)
test_dataset  = TrOCRDataset(combined_dataset["test"],       augment=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Train: 46560 | Val: 1488 | Test: 9301


### Model Setup

In [9]:
from pathlib import Path
from transformers import VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# INPUT_CHECKPOINT_DIR already defined in Cell 1
# Redeclared here for clarity — safe to run again
INPUT_CHECKPOINT_DIR = Path("/kaggle/input/smart-stock-dataset/trocr-smart-stock")

# Load base model from stored best weights
base_model = VisionEncoderDecoderModel.from_pretrained(str(MODEL_INPUT))

# Freeze encoder entirely — ViT visual features preserved from pretraining
# This is the current performance ceiling; unfreeze top 2-4 blocks in future session
for param in base_model.encoder.parameters():
    param.requires_grad = False

# ── Checkpoint resume priority ────────────────────────────────────────────────
# 1. /kaggle/working/trocr-smart-stock  (current session checkpoints, most recent)
# 2. /kaggle/input/.../trocr-smart-stock (uploaded from prior session — fallback)
# 3. Fresh LoRA (no prior checkpoint found)

def find_lora_checkpoints(directory: Path):
    """Find checkpoints with LoRA adapter weights in either format."""
    return sorted([
        ckpt for ckpt in directory.glob("checkpoint-*")
        if (ckpt / "lora_adapter").exists() or (ckpt / "adapter_config.json").exists()
    ])

working_checkpoints = find_lora_checkpoints(CHECKPOINT_DIR)
input_checkpoints   = find_lora_checkpoints(INPUT_CHECKPOINT_DIR) if INPUT_CHECKPOINT_DIR.exists() else []

if working_checkpoints:
    latest = working_checkpoints[-1]
    adapter_path = latest / "lora_adapter" if (latest / "lora_adapter").exists() else latest
    model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=True)
    print(f"Resumed from working dir: {adapter_path}")
elif input_checkpoints:
    latest = input_checkpoints[-1]
    adapter_path = latest / "lora_adapter" if (latest / "lora_adapter").exists() else latest
    model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=True)
    print(f"Resumed from input dir: {adapter_path}")
else:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16,           # rank — increase to 32 in future for more capacity
        lora_alpha=32,  # scaling factor = lora_alpha / r = 2.0
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],  # decoder attention projections only
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    print("No adapter checkpoint found — fresh LoRA applied")

# Required decoder config — must be set on model.config, not model.generation_config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# Generation config — must go on model.generation_config, NOT model.config
model.generation_config.eos_token_id         = processor.tokenizer.sep_token_id
model.generation_config.early_stopping       = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty       = 2.0
model.generation_config.num_beams            = 4

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

trainable params: 1,523,712 || all params: 335,445,504 || trainable%: 0.4542
No adapter checkpoint found — fresh LoRA applied


#### collate_fn

Standalone cell — extracted from the Optuna block so it's always available even when Optuna is commented out.

In [10]:
def collate_fn(batch):
    """
    Custom collator for TrOCR.
    pixel_values and labels are tensors returned by TrOCRDataset.__getitem__.
    torch.stack combines them into batches for the Trainer.
    """
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels       = torch.stack([item["labels"]       for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

### Training Arguments
Block A — OLD CONFIG (fully commented out, do not use)


Kept for reference only. This was the config from Kaggle run 1 (lr=1.695e-4, cosine scheduler). Superseded by Block B below.

In [11]:
# from transformers import Seq2SeqTrainingArguments

# training_args = Seq2SeqTrainingArguments(
#     output_dir=str(CHECKPOINT_DIR),   # /kaggle/working/trocr-smart-stock

#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,

#     # Hardcoded from Optuna best (Trial 3 of latest run)
#     learning_rate=1.695e-4,
#     warmup_ratio=0.0866,
#     weight_decay=0.01,
#     lr_scheduler_type="cosine",
#     max_grad_norm=1.0,

#     eval_strategy="epoch",
#     save_strategy="steps",
#     save_steps=500,
#     load_best_model_at_end=False,
#     save_total_limit=5,

#     predict_with_generate=True,
#     generation_max_length=128,

#     fp16=True,
#     dataloader_num_workers=2,

#     logging_dir=str(WORKING_DIR / "logs"),
#     logging_steps=50,
#     log_level="info",
#     report_to="none",
# )

##### Technique 1 — Force single GPU + fix LR (most important, run first)

DataParallel doubled effective batch size silently. Forcing single GPU makes real training match Optuna's conditions exactly.

##### Block B — ACTIVE CONFIG (Technique 1: single GPU + correct LR)


This is the block that actually runs. os.environ line here is redundant (already set in Cell 1) but harmless.

In [12]:
import os

# Redundant — already set in Cell 1 before all imports.
# Kept here as a safety reminder. Has no effect if torch already imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=6,                # 8 epochs — single GPU converges slower but cleaner
    per_device_train_batch_size=8,     # effective batch = 8 (no DataParallel)
    per_device_eval_batch_size=8,

    # Optuna best from Kaggle run (Trial 0 — CER 0.088 on subset, single GPU)
    learning_rate=1.4824e-4,
    warmup_ratio=0.02672,
    weight_decay=0.01,

    # Cosine with restarts — helps escape local minima mid-training
    # 2 restarts over 8 epochs = restart at epoch ~4
    lr_scheduler_type="cosine_with_restarts",
    lr_scheduler_kwargs={"num_cycles": 2},  # keep — 2 restarts over 6 epochs = restart at epoch ~3

    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=False,
    save_total_limit=5,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=True,                         # mixed precision — required on T4
    dataloader_num_workers=2,

    logging_dir=str(WORKING_DIR / "logs"),
    logging_steps=50,
    log_level="info",
    report_to="none",                  # disable wandb

    # Technique 2 — Gradient Accumulation (COMMENTED OUT)
    # Uncomment if batch 8 causes OOM after encoder unfreezing in a future session.
    # gradient_accumulation_steps=2 gives effective batch 16 without DataParallel.
    # Does NOT invalidate the Optuna LR the way DataParallel does, because
    # accumulation doesn't change the optimizer step rate.
    # gradient_accumulation_steps=2,
)

print(f"Effective batch size: {training_args.per_device_train_batch_size}")
print(f"GPU count visible: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Effective batch size: 8
GPU count visible: 0


### Metrics

In [13]:
!pip install jiwer -q

In [14]:
import numpy as np
from jiwer import cer, wer

def compute_metrics(pred):
    """
    Compute CER and WER for Seq2SeqTrainer.
    Called at end of each eval epoch with generated token ids.
    """
    pred_ids   = pred.predictions
    labels_ids = pred.label_ids

    # pred_ids may be float logits (ndim=3) — argmax to get token ids
    if pred_ids.dtype != np.int64 and pred_ids.ndim == 3:
        pred_ids = np.argmax(pred_ids, axis=-1)

    # Clip to valid vocab range — prevents OverflowError during decode
    vocab_size = processor.tokenizer.vocab_size
    pred_ids   = np.clip(pred_ids, 0, vocab_size - 1)

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 (padding mask) with pad token id before decoding
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    return {
        "cer": round(cer(label_str, pred_str), 4),
        "wer": round(wer(label_str, pred_str), 4),
    }

### Hyperparameter Search with **Optuna**
After the first clean training run with line crops, use Optuna (Bayesian optimization) via HuggingFace built-in hyperparameter_search to find optimal values.

#### Why commented: Current LR (1.4824e-4) was tuned by Optuna on a prior subset run. Running Optuna again costs ~3–4 hours of the 12-hour session before any real training starts.

When to uncomment: After a stable epoch 1 completes on v3 data and CER confirms ~0.088–0.092. If the LR feels off on the new dataset distribution, run Optuna in a separate dedicated session.

Config notes: n_trials=4 (not 10) — 4 trials × 1 epoch on 1/8 of 46,500 = ~5,800 examples per trial ≈ 45 min per trial ≈ 3 hr total. Fits in a dedicated session. predict_with_generate=True kept (unlike old v12 config) — we need CER not just loss to rank trials meaningfully.

In [15]:
# import gc
# import copy
# from transformers import Seq2SeqTrainer
# from peft import LoraConfig, get_peft_model, TaskType

# def optuna_hp_space(trial):
#     return {
#         "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-4, log=True),
#         "warmup_ratio":  trial.suggest_float("warmup_ratio", 0.0, 0.1),
#     }

# def model_init():
#     gc.collect()
#     torch.cuda.empty_cache()

#     m = VisionEncoderDecoderModel.from_pretrained(str(MODEL_INPUT))

#     for param in m.encoder.parameters():
#         param.requires_grad = False

#     lora_config = LoraConfig(
#         task_type=TaskType.SEQ_2_SEQ_LM,
#         r=16,
#         lora_alpha=32,
#         lora_dropout=0.05,
#         target_modules=["q_proj", "v_proj"],
#         bias="none",
#     )
#     m = get_peft_model(m, lora_config)

#     m.config.decoder_start_token_id = processor.tokenizer.cls_token_id
#     m.config.pad_token_id           = processor.tokenizer.pad_token_id
#     m.config.vocab_size             = m.config.decoder.vocab_size
#     m.generation_config.eos_token_id         = processor.tokenizer.sep_token_id
#     m.generation_config.early_stopping       = True
#     m.generation_config.no_repeat_ngram_size = 3
#     m.generation_config.length_penalty       = 2.0
#     m.generation_config.num_beams            = 4
#     return m

# search_args = copy.deepcopy(training_args)
# search_args.num_train_epochs = 1
# search_args.predict_with_generate = True
# search_args.eval_accumulation_steps = 4
# search_args.dataloader_num_workers = 0
# search_args.per_device_train_batch_size = 4
# search_args.per_device_eval_batch_size = 4
# search_args.output_dir = str(WORKING_DIR / "optuna_search")
# search_args.save_strategy = "no"
# search_args.load_best_model_at_end = False
# search_args.eval_strategy = "epoch"
# search_args.logging_steps = 50

# # Use 1/8 of training data for speed — val capped at 200 samples
# search_val_hf = combined_dataset["validation"].select(range(min(200, len(combined_dataset["validation"]))))
# search_val = TrOCRDataset(search_val_hf, augment=False)

# search_hf = combined_dataset["train"].select(range(len(combined_dataset["train"]) // 8))
# search_train = TrOCRDataset(search_hf, augment=True)

# search_trainer = Seq2SeqTrainer(
#     model_init=model_init,
#     args=search_args,
#     train_dataset=search_train,
#     eval_dataset=search_val,
#     data_collator=collate_fn,
#     compute_metrics=compute_metrics,
# )

# best_run = search_trainer.hyperparameter_search(
#     direction="minimize",
#     backend="optuna",
#     hp_space=optuna_hp_space,
#     n_trials=4,
# )

# del search_trainer
# gc.collect()
# torch.cuda.empty_cache()

# print("Best hyperparameters:", best_run.hyperparameters)
# for k, v in best_run.hyperparameters.items():
#     setattr(training_args, k, v)
# training_args.predict_with_generate = True
# print("Updated training_args:", training_args.learning_rate, training_args.warmup_ratio)

### Trainer and Training

In [16]:
from transformers import Seq2SeqTrainer, TrainerCallback
import shutil

class LoRASaveCallback(TrainerCallback):
    """
    Saves LoRA adapter weights alongside each Trainer checkpoint.

    Without this callback, PEFT silently resets adapter weights on checkpoint
    reload — the base model config is restored instead of the adapter state.
    This was Bug #3 in the training history.

    Saves to: checkpoint-{step}/lora_adapter/
    The find_lora_checkpoints() function in Cell 8 looks for this subdir.
    """
    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        adapter_dir = checkpoint_dir / "lora_adapter"
        adapter_dir.mkdir(parents=True, exist_ok=True)
        kwargs["model"].save_pretrained(str(adapter_dir))
        print(f"LoRA adapter saved to {adapter_dir}")
        return control

print(f"LR: {training_args.learning_rate}")
print(f"Warmup ratio: {training_args.warmup_ratio}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

# Remove stale checkpoints from prior bad runs (no lora_adapter subdir)
# These would be picked up by resume logic but have no usable adapter weights
if CHECKPOINT_DIR.exists():
    for ckpt in CHECKPOINT_DIR.glob("checkpoint-*"):
        has_adapter = (ckpt / "lora_adapter").exists()
        if not has_adapter:
            shutil.rmtree(ckpt)
            print(f"Removed stale checkpoint (no LoRA adapter): {ckpt}")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[LoRASaveCallback()],
)

# Resume from latest valid checkpoint if available
valid_checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"))
resume_from = str(valid_checkpoints[-1]) if valid_checkpoints else None

if resume_from:
    print(f"Resuming from: {resume_from}")
else:
    print("Starting fresh training")

trainer.train(resume_from_checkpoint=resume_from)

LR: 0.00014824
Warmup ratio: 0.02672
Epochs: 8
Batch size: 8


***** Running training *****
  Num examples = 46,560
  Num Epochs = 8
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 46,560
  Number of trainable parameters = 1,523,712


Starting fresh training


Epoch,Training Loss,Validation Loss,Cer,Wer
1,1.605225,1.210828,0.378800,0.703200


Saving model checkpoint to /kaggle/working/trocr-smart-stock/checkpoint-500
loading configuration file /kaggle/input/datasets/maazahmad69/trocr-smart-stock-best/content/drive/MyDrive/SmartStock/trocr-smart-stock-best/config.json
Model config VisionEncoderDecoderConfig {
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": 768,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 12,
    "decoder_start_token_id": 2,
    "dropout": 0.1,
    "dtype": "float32",
    "eos_token_id": 2,
    "finetuning_task": null,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "init

...

Saving model checkpoint to /kaggle/working/trocr-smart-stock/checkpoint-7500
loading configuration file /kaggle/input/datasets/maazahmad69/trocr-smart-stock-best/content/drive/MyDrive/SmartStock/trocr-smart-stock-best/config.json
Model config VisionEncoderDecoderConfig {
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": 768,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 12,
    "decoder_start_token_id": 2,
    "dropout": 0.1,
    "dtype": "float32",
    "eos_token_id": 2,
    "finetuning_task": null,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "ini

LoRA adapter saved to /kaggle/working/trocr-smart-stock/checkpoint-7500/lora_adapter


Saving model checkpoint to /kaggle/working/trocr-smart-stock/checkpoint-8000
loading configuration file /kaggle/input/datasets/maazahmad69/trocr-smart-stock-best/content/drive/MyDrive/SmartStock/trocr-smart-stock-best/config.json
Model config VisionEncoderDecoderConfig {
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": 768,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 12,
    "decoder_start_token_id": 2,
    "dropout": 0.1,
    "dtype": "float32",
    "eos_token_id": 2,
    "finetuning_task": null,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "ini

LoRA adapter saved to /kaggle/working/trocr-smart-stock/checkpoint-8000/lora_adapter


KeyboardInterrupt: 

### Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
print(history.columns)
history.head()

In [ ]:
train_logs = history[history["loss"].notna()]
eval_logs  = history[history["eval_loss"].notna()]

plt.figure(figsize=(12,6))
plt.plot(train_logs["step"], train_logs["loss"], label="Training Loss")
plt.plot(eval_logs["step"], eval_logs["eval_loss"], label="Validation Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("TrOCR Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))
axes[0].plot(eval_logs["epoch"], eval_logs["eval_cer"], marker="o")
axes[0].set_title("CER")
axes[0].set_xlabel("Epoch")
axes[1].plot(eval_logs["epoch"], eval_logs["eval_wer"], marker="o")
axes[1].set_title("WER")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

In [ ]:
summary = eval_logs[["epoch", "eval_loss", "eval_cer", "eval_wer"]]
summary

### Save & Export

In [ ]:
from peft import PeftModel

# Merge LoRA weights into base model — produces a standard VisionEncoderDecoderModel
# with no PEFT dependency. Required for inference without peft installed.
merged_model = model.merge_and_unload()

for save_path in [str(BEST_MODEL_DIR)]:
    merged_model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    merged_model.generation_config.save_pretrained(save_path)
    print(f"Saved to: {save_path}")

expected_files = [
    "model.safetensors", "config.json", "generation_config.json",
    "tokenizer_config.json", "tokenizer.json", "processor_config.json",
]
print(f"\nVerification ({BEST_MODEL_DIR}):")
for fname in expected_files:
    fpath = BEST_MODEL_DIR / fname
    exists = fpath.exists()
    size = fpath.stat().st_size / 1e6 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname} ({size:.1f} MB)")

### Evaluate on Test Set

In [ ]:
from jiwer import cer, wer

model.eval()
all_preds, all_labels = [], []
skipped = 0
first_error_printed = False

for idx, sample in enumerate(combined_dataset["test"]):
    try:
        image = Image.open(io.BytesIO(sample["image_bytes"])).convert("RGB")
        w, h = image.size
        if w < 4 or h < 4:
            skipped += 1
            continue

        pixel_values = processor(
            images=image, return_tensors="pt"
        ).pixel_values.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                max_new_tokens=128,
            )

        pred = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0]

        all_preds.append(pred)
        all_labels.append(sample["text"])

    except Exception as e:
        if not first_error_printed:
            print(f"First exception at idx={idx}: {type(e).__name__}: {e}")
            first_error_printed = True
        skipped += 1

if all_preds:
    test_cer = cer(all_labels, all_preds)
    test_wer = wer(all_labels, all_preds)
    print(f"Test CER : {test_cer:.4f}")
    print(f"Test WER : {test_wer:.4f}")
else:
    print("No predictions collected — all samples skipped or errored")

print(f"Skipped  : {skipped} / {len(combined_dataset['test'])}")
print(f"Evaluated: {len(all_preds)} / {len(combined_dataset['test'])}")

# Targets: CER ≤ 0.05, WER ≤ 0.10

### Qualitative Evaluation

In [ ]:
import io
from PIL import Image

sample = combined_dataset["test"][0]
image = Image.open(io.BytesIO(sample["image_bytes"])).convert("RGB")

pixel_values = processor(
    image,
    return_tensors="pt"
).pixel_values.to(model.device)

generated_ids = model.generate(pixel_values=pixel_values, max_new_tokens=128)

prediction = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("GROUND TRUTH:")
print(sample["text"])
print("")
print("="*80)
print("PREDICTION:")
print(prediction)

image